# Student Depression Prediction & Exploratory Data Analysis (EDA)
### Comprehensive End-to-End Analysis, Feature Engineering, and Machine Learning Modeling

**Objective:** Analyze the psychological, social, academic, and health indicators of university students to discover patterns associated with depression, evaluate multiple classification algorithms, and build an interpretable risk prediction pipeline.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Dataset Loading and Initial Inspection
We load the verified `student_depression_dataset.csv`.

In [ ]:
dataset_path = '../../dataset/student_depression_dataset.csv'
if not os.path.exists(dataset_path):
    dataset_path = '../dataset/student_depression_dataset.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'Student_Depression_Dataset.csv'

df = pd.read_csv(dataset_path)
print("Shape:", df.shape)
display(df.head())
display(df.info())
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

## 2. Exploratory Data Analysis (EDA)
### 2.1 Target Variable Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
counts = df['Depression'].value_counts()
sns.barplot(x=counts.index, y=counts.values, palette=['#3b82f6', '#ef4444'], ax=ax)
ax.set_title("Distribution of Student Depression Label", fontsize=14, fontweight='bold')
ax.set_ylabel("Number of Students")
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{int(height)} ({height/len(df)*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=12)
plt.show()

### 2.2 Health Condition vs Depression
Health condition shows the strongest association with depression.

In [ ]:
order = ['Good', 'Normal', 'Abnormal', 'Critical']
plt.figure(figsize=(9, 5))
sns.countplot(data=df, x='Health_condition', hue='Depression', order=order, palette=['#10b981', '#f43f5e'])
plt.title("Health Condition vs Student Depression", fontsize=14, fontweight='bold')
plt.xlabel("Reported Health Condition")
plt.ylabel("Count")
plt.legend(title='Depression')
plt.show()

### 2.3 Social and Relational Factors
Analyzing Friend Circles, Living Style, and Relationship Status.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x='Relationship_status', hue='Depression', ax=axes[0], palette=['#60a5fa', '#f87171'])
axes[0].set_title("Relationship Status vs Depression", fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

sns.countplot(data=df, x='friend_circle', hue='Depression', ax=axes[1], palette=['#60a5fa', '#f87171'])
axes[1].set_title("Active Friend Circle vs Depression", fontweight='bold')

sns.countplot(data=df, x='Living_style', hue='Depression', ax=axes[2], palette=['#60a5fa', '#f87171'])
axes[2].set_title("Living Style (Bachelor vs Family)", fontweight='bold')

plt.tight_layout()
plt.show()

### 2.4 Academic Indicators (CGPA & Semester)
Examining student academic achievement across depression status.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Depression', y='CGPA', ax=axes[0], palette=['#6ee7b7', '#fda4af'])
axes[0].set_title("CGPA Distribution by Depression Status", fontweight='bold')

sns.histplot(data=df, x='Cost_without_study', hue='Depression', kde=True, ax=axes[1], palette=['#3b82f6', '#ef4444'])
axes[1].set_title("Non-Academic Living Cost Distribution", fontweight='bold')

plt.tight_layout()
plt.show()

### 2.5 Correlation Heatmap of Numerical Features

In [ ]:
num_df = df[['Age', 'Current_semester', 'CGPA', 'Cost_without_study']].copy()
num_df['Depression_numeric'] = (df['Depression'] == 'Yes').astype(int)

plt.figure(figsize=(8, 6))
sns.heatmap(num_df.corr(), annot=True, cmap='coolwarm', fmt=".3f", cbar=True)
plt.title("Correlation Heatmap of Numerical Features", fontsize=14, fontweight='bold')
plt.show()

## 3. Machine Learning Modeling & Comparison
We evaluate Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting using 5-Fold Stratified Cross-Validation and holdout test evaluation.

In [ ]:
num_features = ['Age', 'Current_semester', 'CGPA', 'Cost_without_study']
cat_features = ['Gender', 'Family_background', 'Relationship_status', 'Marital_status', 'Living_style', 'friend_circle', 'Health_condition', 'Home_division']

X = df[num_features + cat_features]
y = (df['Depression'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore'), cat_features)
    ]
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=10, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42)
}

results = []
for name, clf in models.items():
    pipe = Pipeline(steps=[('preproc', preprocessor), ('clf', clf)])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'Test Accuracy': accuracy_score(y_test, y_pred),
        'Test Recall': recall_score(y_test, y_pred),
        'Test Precision': precision_score(y_test, y_pred),
        'Test F1': f1_score(y_test, y_pred),
        'Test ROC-AUC': roc_auc_score(y_test, y_prob),
        'CV F1 Mean': cv_res['test_f1'].mean()
    })

res_df = pd.DataFrame(results).sort_values('Test F1', ascending=False)
display(res_df)

## 4. Conclusion & Key Findings
1. **Dominant Predictor:** `Health_condition` is the single strongest indicator of student depression.
2. **Social Buffers:** Having an active friend circle and being in a stable relationship correlate with significantly lower rates of depression.
3. **Academic Factor:** Slightly higher CGPA is inversely associated with depression, reflecting the stress of academic struggle.
4. **Best Classifier:** Gradient Boosting achieved >0.98 F1-score and >0.99 ROC-AUC, effectively balancing precision and recall.